# 06 — UMAP Visualizations (all 4 models)

Loads `vanilla_ae_z64`, `ae_z64`, `ae_z64_w1`, `dec_z64_k21` checkpoints, encodes a 15K subsample, computes UMAP per-model, and saves:
- 12 individual plots (4 models × 3 axes: genre / decade / lang)
- 1 architecture comparison panel

Output: `MyDrive/CineEmbed/artifacts/figures/umap/`. Total runtime: **~6-8 min on T4**.

## Setup — Drive mount + git pull + install

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Auth: try GitHub token from Colab Secrets, fall back to public URL
    try:
        from google.colab import userdata
        _token = userdata.get('GITHUB_TOKEN')
        REPO_URL = f"https://{_token}@github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    except Exception:
        REPO_URL = "https://github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"

    REPO_ROOT = Path('/content/cineembed-repo')   # NOT 'cineembed' (namespace shadow)
    ARTIFACTS = Path('/content/drive/MyDrive/CineEmbed/artifacts')

    if not REPO_ROOT.exists():
        get_ipython().system(f'git clone {REPO_URL} {REPO_ROOT}')
    else:
        get_ipython().system(f'cd {REPO_ROOT} && git pull -q')

    get_ipython().system(f'pip install -e {REPO_ROOT} -q')
    get_ipython().system('pip install umap-learn -q')

    # Ensure src is importable
    sys.path.insert(0, str(REPO_ROOT / 'src'))
else:
    # Local fallback (mac dev)
    REPO_ROOT = Path(__file__).parent.parent if '__file__' in globals() else Path.cwd().parent
    ARTIFACTS = REPO_ROOT / 'artifacts'
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print('REPO_ROOT:', REPO_ROOT)
print('ARTIFACTS:', ARTIFACTS)
print('artifacts exists:', ARTIFACTS.exists())
print('feature_matrix.npz exists:', (ARTIFACTS / 'feature_matrix.npz').exists())
print('models exist:', sorted([p.name for p in (ARTIFACTS / 'models').glob('*.pt')]))

## UMAP — encode + project + plot for all 4 models

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import umap

from cineembed import data, backbone, heads, train

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODELS_DIR = ARTIFACTS / 'models'
FIGS_DIR = ARTIFACTS / 'figures' / 'umap'
FIGS_DIR.mkdir(parents=True, exist_ok=True)
print(f'device={DEVICE}, figures dir={FIGS_DIR}')

# --- Load data + labels ----------------------------------------------
print(f"\n[{time.strftime('%H:%M:%S')}] loading feature matrix + labels...")
X, feature_names = data.load_feature_matrix(ARTIFACTS / 'feature_matrix.npz')
block_slices = data.get_block_indices(feature_names)
labels_full = data.get_labels(ARTIFACTS / 'movies_eda_final.csv')

BLOCK_DIMS = {b: slc.stop - slc.start for b, slc in block_slices.items()}
PROJ_DIMS = dict(backbone.DEFAULT_PROJ_DIMS)
print('BLOCK_DIMS:', BLOCK_DIMS)
print('PROJ_DIMS: ', PROJ_DIMS)

N_VIS = 15000
rng = np.random.default_rng(42)
vis_idx = rng.choice(X.shape[0], size=N_VIS, replace=False)
labels_vis = {k: v[vis_idx] for k, v in labels_full.items()}
print(f"[{time.strftime('%H:%M:%S')}] visualizing {N_VIS:,} films (of {X.shape[0]:,})")


# --- Model builders --------------------------------------------------
def build_ae_model(z_dim, vanilla=False):
    if vanilla:
        bb_fc = nn.Sequential(
            nn.Linear(564, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, z_dim),
        )
        class _VanillaWrap(nn.Module):
            def __init__(self, fc, z_dim_):
                super().__init__()
                self.fc = fc
                self.latent_dim = z_dim_
                self.block_order = list(BLOCK_DIMS.keys())
            def forward(self, blocks, block_mask=None):
                X_cat = torch.cat([blocks[b] for b in self.block_order], dim=1)
                return self.fc(X_cat)
        bb = _VanillaWrap(bb_fc, z_dim)
    else:
        bb = backbone.MultiModalBackbone(BLOCK_DIMS, PROJ_DIMS, hidden_dim=128, latent_dim=z_dim)
    return heads.AEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)


def build_dec_model(z_dim, k):
    bb = backbone.MultiModalBackbone(BLOCK_DIMS, PROJ_DIMS, hidden_dim=128, latent_dim=z_dim)
    ae_head = heads.AEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)
    return heads.DECHead(bb, ae_head.decoder, n_clusters=k, latent_dim=z_dim)


# --- Encode subsample ------------------------------------------------
def encode_subsample(model, X, vis_idx, block_slices, batch_size=2048):
    model.eval()
    X_sub = X[vis_idx]
    out = []
    with torch.no_grad():
        for i in range(0, len(vis_idx), batch_size):
            xb = X_sub[i:i+batch_size].to(DEVICE)
            blocks = {b: xb[:, slc] for b, slc in block_slices.items()}
            z = model.backbone(blocks) if hasattr(model, 'backbone') else model.encode(blocks)
            out.append(z.cpu().numpy())
    return np.concatenate(out, axis=0)


# --- Plot helper -----------------------------------------------------
def plot_umap(z2d, labels, title, savepath, top_n=12, figsize=(9, 7)):
    labels = np.asarray(labels).astype(str)
    counts = {}
    for v in labels:
        counts[v] = counts.get(v, 0) + 1
    top = sorted(counts, key=counts.get, reverse=True)[:top_n]
    plot_labels = np.where(np.isin(labels, top), labels, 'Other')

    fig, ax = plt.subplots(figsize=figsize)
    cmap = plt.get_cmap('tab20', len(top) + 1)

    if 'Other' in set(plot_labels):
        m = plot_labels == 'Other'
        ax.scatter(z2d[m, 0], z2d[m, 1], s=2, c=[(0.7, 0.7, 0.7, 0.3)], label=f'Other ({m.sum()})')
    for i, k in enumerate(top):
        m = plot_labels == k
        ax.scatter(z2d[m, 0], z2d[m, 1], s=3, color=cmap(i), alpha=0.7, label=f'{k} ({m.sum()})')

    ax.set_title(title, fontsize=13)
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8, markerscale=3)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.savefig(savepath, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'  saved: {savepath.name}')


# --- Run UMAP for each model ----------------------------------------
runs = [
    ('vanilla_ae_z64',  lambda: build_ae_model(64, vanilla=True)),
    ('ae_z64',          lambda: build_ae_model(64, vanilla=False)),
    ('ae_z64_w1',       lambda: build_ae_model(64, vanilla=False)),
    ('dec_z64_k21',     lambda: build_dec_model(64, 21)),
]

z2d_cache = {}

for run_name, model_builder in runs:
    print(f"\n[{time.strftime('%H:%M:%S')}] === {run_name} ===")
    ckpt = MODELS_DIR / f'{run_name}.pt'
    if not ckpt.exists():
        print(f'  SKIP — checkpoint missing: {ckpt}')
        continue

    model = model_builder().to(DEVICE)
    train.load_checkpoint(model, ckpt, device=DEVICE)

    print(f'  encoding {N_VIS:,} films...')
    t0 = time.time()
    z = encode_subsample(model, X, vis_idx, block_slices)
    print(f'    encoded in {time.time()-t0:.1f}s, z.shape={z.shape}')

    print(f'  computing UMAP...')
    t0 = time.time()
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, metric='cosine', random_state=42)
    z2d = reducer.fit_transform(z)
    print(f'    UMAP done in {time.time()-t0:.1f}s')
    z2d_cache[run_name] = z2d

    for axis_name, label_key in [('genre', 'primary_genre'), ('decade', 'decade_bin'), ('lang', 'lang_top10')]:
        plot_umap(
            z2d, labels_vis[label_key],
            title=f'{run_name} — colored by {axis_name}',
            savepath=FIGS_DIR / f'umap_{run_name}_{axis_name}.png',
        )


# --- Comparison panel: vanilla vs ae_z64 vs DEC, by genre -----------
print(f"\n[{time.strftime('%H:%M:%S')}] === Comparison panel (genre) ===")
panel_runs = [r for r in ['vanilla_ae_z64', 'ae_z64', 'dec_z64_k21'] if r in z2d_cache]
if len(panel_runs) >= 2:
    fig, axes = plt.subplots(1, len(panel_runs), figsize=(7 * len(panel_runs), 6))
    if len(panel_runs) == 1:
        axes = [axes]
    genre_vis = labels_vis['primary_genre']
    counts = {v: int((genre_vis == v).sum()) for v in np.unique(genre_vis)}
    top = sorted(counts, key=counts.get, reverse=True)[:12]
    cmap = plt.get_cmap('tab20', 13)
    plot_labels = np.where(np.isin(genre_vis, top), genre_vis, 'Other')

    for ax, run_name in zip(axes, panel_runs):
        z2d = z2d_cache[run_name]
        m_other = plot_labels == 'Other'
        ax.scatter(z2d[m_other, 0], z2d[m_other, 1], s=2, c=[(0.7, 0.7, 0.7, 0.3)])
        for i, k in enumerate(top):
            m = plot_labels == k
            ax.scatter(z2d[m, 0], z2d[m, 1], s=3, color=cmap(i), alpha=0.7, label=k)
        ax.set_title(run_name, fontsize=12)
        ax.set_xticks([])
        ax.set_yticks([])

    handles, labels_ = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels_, loc='center right', bbox_to_anchor=(1.08, 0.5),
               fontsize=9, markerscale=3)
    fig.suptitle('Architecture comparison — UMAP latent, colored by primary_genre', fontsize=14)
    plt.tight_layout()
    panel_path = FIGS_DIR / 'umap_comparison_genre.png'
    plt.savefig(panel_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'  saved: {panel_path.name}')

# Final summary — separate glob from f-string to avoid backslash-in-expression issues
n_files = len(list(FIGS_DIR.glob('*.png')))
print(f"\n[{time.strftime('%H:%M:%S')}] DONE")
print(f'  total files in {FIGS_DIR}: {n_files}')